# Modeling — next-day direction, chronological evaluation

Run `python scripts/build_features.py` first (needs `data/raw/sp500_panel.csv` from `scripts/download_data.py`).

**MLU concepts applied:** chronological (not random) train/test split for time series; comparing against naive baselines before trusting any model; regularized linear model + shallow bagged trees as the "start simple" baseline models per the tabular-data material, before reaching for anything more complex.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
from stock_predictor.model import (
    chronological_split, naive_baselines, build_models, evaluate, get_X_y, time_series_cv_scores,
)

feat = pd.read_csv('../data/processed_features.csv', parse_dates=['date'])
train_df, test_df = chronological_split(feat, test_frac=0.2)
print(f"train: {train_df.date.min().date()} -> {train_df.date.max().date()}  ({len(train_df):,} rows)")
print(f"test:  {test_df.date.min().date()} -> {test_df.date.max().date()}  ({len(test_df):,} rows)")

train: 2016-02-02 -> 2024-05-23  (1,011,331 rows)
test:  2024-05-24 -> 2026-06-29  (261,459 rows)


## Naive baselines (the bar the model must clear)

- **Majority class**: always predict the more common direction from the training period.
- **Persistence**: predict tomorrow moves the same direction as today's realized return.

In [2]:
baselines = naive_baselines(train_df, test_df)
baselines

{'majority_class_baseline': 0.5189494337544318,
 'persistence_baseline': 0.4912892652385269}

## Time-aware cross-validation (model selection on the training period only)

Expanding-window `TimeSeriesSplit` on the training data — never touches the held-out test period. Used here to sanity-check stability before touching the test set at all.

In [3]:
X_train, y_train = get_X_y(train_df)
X_test, y_test = get_X_y(test_df)

models = build_models()
cv_scores = time_series_cv_scores(models['logistic_regression'], X_train, y_train, n_splits=5)
print('Logistic regression, 5-fold expanding-window CV accuracy:', [round(s, 4) for s in cv_scores])
print('mean:', round(sum(cv_scores) / len(cv_scores), 4), ' vs. majority-class baseline:', round(baselines['majority_class_baseline'], 4))

Logistic regression, 5-fold expanding-window CV accuracy: [0.5268, 0.5269, 0.5294, 0.5274, 0.525]
mean: 0.5271  vs. majority-class baseline: 0.5189


## Final chronological holdout evaluation

Fit fresh on the *entire* training period, evaluate once on the untouched test period (2024-05-24 onward). Random forest is capped at `max_depth=6` and a large `min_samples_leaf` deliberately — deep unconstrained trees overfit daily return noise almost immediately.

In [4]:
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = evaluate(model, X_test, y_test)

results_df = pd.DataFrame(results).T
results_df.loc['majority_class_baseline', 'accuracy'] = baselines['majority_class_baseline']
results_df.loc['persistence_baseline', 'accuracy'] = baselines['persistence_baseline']
results_df

,accuracy,precision,recall,f1,roc_auc
logistic_regression,0.518686,0.519816,0.951181,0.672251,0.507168
random_forest,0.518655,0.519361,0.971920,0.676971,0.510112
majority_class_baseline,0.518949,NaN,NaN,NaN,NaN
persistence_baseline,0.491289,NaN,NaN,NaN,NaN


## Honest conclusion

Both models land within ~0.03pp of the majority-class baseline (~51.9%) and ROC-AUC barely clears 0.50 (0.507-0.510) — essentially no discriminative power beyond chance, and **neither model beats the naive baselines** on the held-out chronological test set. This is the expected, honest result for daily direction prediction from price/volume technical features alone, consistent with the near-zero return autocorrelation found in the EDA notebook (efficient-market behavior). Recall is high (~0.95-0.97) only because both models lean toward predicting "up" — precision stays near the base rate, so this is not a usable trading signal.

Logistic regression is used as the served model (Phase 4) purely for speed/simplicity/interpretability given the tie in performance — not because it "won." This honest negative result, not a fabricated edge, is the point of the project: a defensible pipeline that reports what the data actually supports.

In [5]:
import json
import joblib
from stock_predictor.features import FEATURE_COLUMNS

final_model = models['logistic_regression']
final_model.fit(X_train, y_train)  # refit on full train_df (already done above, kept explicit)
joblib.dump(final_model, '../models/direction_model.joblib')

meta = {
    'feature_columns': FEATURE_COLUMNS,
    'train_date_range': [str(train_df.date.min().date()), str(train_df.date.max().date())],
    'test_date_range': [str(test_df.date.min().date()), str(test_df.date.max().date())],
    'n_train_rows': int(len(train_df)),
    'n_test_rows': int(len(test_df)),
    'baselines': baselines,
    'test_metrics': {k: results['logistic_regression'][k] for k in results['logistic_regression']},
    'random_forest_test_metrics': {k: results['random_forest'][k] for k in results['random_forest']},
    'logreg_timeseries_cv_accuracy': cv_scores,
}
json.dump(meta, open('../models/metrics.json', 'w'), indent=2)
print('Saved models/direction_model.joblib and models/metrics.json')

Saved models/direction_model.joblib and models/metrics.json
